# Fix Data Leakage: Repartition at Original ID Level

In [2]:
import os
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

DATASET_ROOT = Path('Dataset')
SPLIT_ROOT = Path('Dataset_Split')
SEED = 42

np.random.seed(SEED)

In [3]:
os.chdir("../../")
os.getcwd()

'c:\\Users\\Ansh Lulla\\VS-Code\\PlantDiseaseProject\\plant-disease-project'

## Extract Original IDs & Group Augmentations

In [4]:
class_folders = sorted([d for d in DATASET_ROOT.iterdir() if d.is_dir()])

id_to_images = defaultdict(lambda: defaultdict(list))

for class_dir in class_folders:
    class_name = class_dir.name
    images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg'))
    
    for img_path in images:
        filename = img_path.name
        original_id = filename.split('___')[0]
        id_to_images[original_id][class_name].append(img_path)

total_ids = len(id_to_images)
print(f"Total unique original IDs: {total_ids}")
print(f"Total images: {sum(len(imgs) for class_dict in id_to_images.values() for imgs in class_dict.values())}")

sample_id = list(id_to_images.keys())[0]
print(f"\nSample ID {sample_id}:")
for cls, paths in id_to_images[sample_id].items():
    print(f"  {cls}: {len(paths)} augmentations")

Total unique original IDs: 28077
Total images: 35725

Sample ID 00075aa8-d81a-4184-8541-b692b78d398a:
  Apple___Apple_scab: 1 augmentations


## Split IDs into Train/Val/Test

In [5]:
all_ids = np.array(list(id_to_images.keys()))
np.random.shuffle(all_ids)

n_train = int(len(all_ids) * 0.8)
n_val = int(len(all_ids) * 0.1)

train_ids = set(all_ids[:n_train])
val_ids = set(all_ids[n_train:n_train + n_val])
test_ids = set(all_ids[n_train + n_val:])

print(f"Train IDs: {len(train_ids)}")
print(f"Val IDs: {len(val_ids)}")
print(f"Test IDs: {len(test_ids)}")
print(f"\nVerify no overlap: {len(train_ids & val_ids & test_ids) == 0}")

Train IDs: 22461
Val IDs: 2807
Test IDs: 2809

Verify no overlap: True


## Create Split Directories & Copy Files

In [6]:
for split in ['train', 'val', 'test']:
    split_dir = SPLIT_ROOT / split
    split_dir.mkdir(parents=True, exist_ok=True)
    
    for class_dir in class_folders:
        class_split_dir = split_dir / class_dir.name
        class_split_dir.mkdir(parents=True, exist_ok=True)

print(f"Created directory structure at {SPLIT_ROOT}")

Created directory structure at Dataset_Split


## Assign Images to Splits

In [7]:
split_counts = {'train': 0, 'val': 0, 'test': 0}

for original_id, class_dict in id_to_images.items():
    if original_id in train_ids:
        split = 'train'
    elif original_id in val_ids:
        split = 'val'
    else:
        split = 'test'
    
    for class_name, img_paths in class_dict.items():
        dst_dir = SPLIT_ROOT / split / class_name
        for img_path in img_paths:
            dst_path = dst_dir / img_path.name
            shutil.copy2(img_path, dst_path)
            split_counts[split] += 1

print("\nImages copied to splits:")
for split, count in split_counts.items():
    pct = (count / sum(split_counts.values())) * 100
    print(f"  {split}: {count} ({pct:.1f}%)")


Images copied to splits:
  train: 28627 (80.1%)
  val: 3551 (9.9%)
  test: 3547 (9.9%)


## Verify Split Class Distributions

In [8]:
split_class_counts = {}

for split in ['train', 'val', 'test']:
    split_class_counts[split] = {}
    split_dir = SPLIT_ROOT / split
    
    for class_dir in (split_dir).iterdir():
        if class_dir.is_dir():
            images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg'))
            split_class_counts[split][class_dir.name] = len(images)

print("\nClass distribution per split:")
for split in ['train', 'val', 'test']:
    total = sum(split_class_counts[split].values())
    print(f"\n{split.upper()} ({total} images):")
    for cls, count in sorted(split_class_counts[split].items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {cls}: {count}")


Class distribution per split:

TRAIN (28627 images):
  Tomato__Tomato_YellowLeaf__Curl_Virus: 2589
  Tomato_Bacterial_spot: 1705
  Apple___healthy: 1607
  Apple___Apple_scab: 1605
  Apple___Black_rot: 1575

VAL (3551 images):
  Tomato__Tomato_YellowLeaf__Curl_Virus: 322
  Apple___Apple_scab: 216
  Tomato_Bacterial_spot: 212
  Apple___Black_rot: 208
  Tomato_Late_blight: 198

TEST (3547 images):
  Tomato__Tomato_YellowLeaf__Curl_Virus: 297
  Tomato_Bacterial_spot: 210
  Tomato_Late_blight: 209
  Corn_(maize)___Northern_Leaf_Blight: 205
  Apple___Black_rot: 204


## Leakage Verification

In [11]:
def extract_original_ids(split_dir):
    ids = set()
    for class_dir in split_dir.iterdir():
        if class_dir.is_dir():
            for img_path in class_dir.glob('*'):
                if img_path.is_file():
                    original_id = img_path.name.split('___')[0]
                    ids.add(original_id)
    return ids

train_ids_check = extract_original_ids(SPLIT_ROOT / 'train')
val_ids_check = extract_original_ids(SPLIT_ROOT / 'val')
test_ids_check = extract_original_ids(SPLIT_ROOT / 'test')

train_val_overlap = len(train_ids_check & val_ids_check)
train_test_overlap = len(train_ids_check & test_ids_check)
val_test_overlap = len(val_ids_check & test_ids_check)

print("\nLEAKAGE VERIFICATION:")
print(f"Train-Val overlap: {train_val_overlap}")
print(f"Train-Test overlap: {train_test_overlap}")
print(f"Val-Test overlap: {val_test_overlap}")
print(f"\nResult: {'✓ ZERO LEAKAGE' if (train_val_overlap + train_test_overlap + val_test_overlap) == 0 else '✗ LEAKAGE DETECTED'}")


LEAKAGE VERIFICATION:
Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0

Result: ✓ ZERO LEAKAGE


## Generate Split Summary CSV

In [10]:
summary_data = []

for split in ['train', 'val', 'test']:
    for cls, count in split_class_counts[split].items():
        summary_data.append({
            'Split': split,
            'Class': cls,
            'Count': count
        })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(SPLIT_ROOT / 'split_summary.csv', index=False)

print("Summary saved to split_summary.csv")
print(f"\nTotal images in split: {len(summary_df) // 3} classes × 3 splits")

Summary saved to split_summary.csv

Total images in split: 23 classes × 3 splits


## Final Report

In [13]:
train_total = sum(split_class_counts['train'].values())
val_total = sum(split_class_counts['val'].values())
test_total = sum(split_class_counts['test'].values())
grand_total = train_total + val_total + test_total

print("\n" + "="*60)
print("DATA REPARTITIONING COMPLETE")
print("="*60)
print(f"\nTotal images: {grand_total}")
print(f"  Train: {train_total} ({train_total/grand_total*100:.1f}%)")
print(f"  Val: {val_total} ({val_total/grand_total*100:.1f}%)")
print(f"  Test: {test_total} ({test_total/grand_total*100:.1f}%)")
print(f"\nTotal classes: {len(split_class_counts['train'])}")
print(f"Original image IDs used: {len(train_ids | val_ids | test_ids)}")
print(f"\nLeakage status: {'✓ ZERO LEAKAGE' if (train_val_overlap + train_test_overlap + val_test_overlap) == 0 else '✗ LEAKAGE DETECTED'}")
print(f"\nOutput directory: {SPLIT_ROOT}")


DATA REPARTITIONING COMPLETE

Total images: 35725
  Train: 28627 (80.1%)
  Val: 3551 (9.9%)
  Test: 3547 (9.9%)

Total classes: 23
Original image IDs used: 28077

Leakage status: ✓ ZERO LEAKAGE

Output directory: Dataset_Split
